## Libraries

In [ ]:
from pyspark.sql.functions import col, lit, date_format, from_unixtime, avg
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from py4j.protocol import Py4JJavaError
from pyspark.sql import SparkSession

In [ ]:
# Instance SparkSession
spark = SparkSession.builder\
        .appName('Transforms Tables')\
        .master('local[*]')\
        .getOrCreate()

# Extract Stage

## Read inputs

In [ ]:
# Define dictionary for input_paths
paths_list = [
    'data/raw/cryptos',
    'data/raw/market_bitcoin',
    'data/raw/historical_bitcoin_prices'
]

def read_paths(paths):
    try:
        # Read CSV files into PySpark DataFrames
        cryptos_df = spark.read.csv(path=paths[0], header=True, sep=',')
        market_bitcoin_df = spark.read.csv(path=paths[1], header=True, sep=',')
        historical_bitcoin_prices_df = spark.read.csv(path=paths[2], header=True, sep=',')
    
        # Save DataFrames in a dictionary
        df_collection_dict = {
            'cryptos': cryptos_df,
            'market_bitcoin': market_bitcoin_df,
            'historical_bitcoin_prices': historical_bitcoin_prices_df
        }
    
        return df_collection_dict

    except py4j.protocol.Py4JJavaError as err:
        if 'org.apache.hadoop.security.AccessControlException' in str(err.java_exception):
            print('Permission denied: User does not have access to the specified file or directory.')
        else:
            print('An unexpected error occurred:', str(err))

    except AnalysisException as err:
        if 'Permission denied' in str(err):
            print('Permission denied: User does not have access to the specified file or directory.')
        elif 'cannot resolve' in str(err):
            print('Column not found: The specified column does not exist in the table.')
        elif 'Table or view not found' in str(err):
            print('Table not found: The specified table or view does not exist.')
        else:
            print('An AnalysisException occurred:', str(err))

    except Exception as err:
        print('An unexpected error occurred:', str(err))

    return None

In [ ]:
# Instance read_paths
dataframes_dict = read_paths(paths=paths_list)

# Transform Stage

## Change schema and apply Transforms

In [ ]:
def change_schema_and_transforms(df1, df2):
    try:
        # Define columns for change in df1
        market_bitcoin_column_types = {
            'current_price': 'int',
            'market_cap': 'int',
            'total_volume': 'int'
        }
    
        # Define columns for change in df2
        historical_bitcoin_prices_column_types = {
            'prices': 'double',
            'market_caps': 'double',
            'total_volumes': 'double'
        }
        
        
        # Apply changes in df1
        market_bitcoin_transformed = df1.select(
            col('crypto_id'),
            col('currency'),
            *[col(c).cast(t) for c, t in market_bitcoin_column_types.items()]
        )

        # Apply changes in df2
        historical_bitcoin_prices_transformed = df2.select(
            date_format(from_unixtime(col('timestamp') / 1000), 'yyyy-MM-dd').alias('date'),
            *[col(c).cast(t) for c, t in historical_bitcoin_prices_column_types.items()]
        )

        # Save DataFrames in a Dictionary
        df_transf_collection_dict = {
            'market_bitcoin_transf': market_bitcoin_transformed,
            'historical_bitcoin_prices_transf' : historical_bitcoin_prices_transformed
        }
        
        return df_transf_collection_dict

    except py4j.protocol.Py4JJavaError as err:
        if 'org.apache.hadoop.security.AccessControlException' in str(err.java_exception):
            print('Permission denied: User does not have access to the specified file or directory.')
        else:
            print('An unexpected error occurred:', str(err))

    except AnalysisException as err:
        if 'Permission denied' in str(err):
            print('Permission denied: User does not have access to the specified file or directory.')
        elif 'cannot resolve' in str(err):
            print('Column not found: The specified column does not exist in the table.')
        elif 'Table or view not found' in str(err):
            print('Table not found: The specified table or view does not exist.')
        else:
            print('An AnalysisException occurred:', str(err))

    except Exception as err:
        print('An unexpected error occurred:', str(err))

    return None

In [ ]:
df_transformed = change_schema_and_transforms(df1= dataframes_dict['market_bitcoin'], df2= dataframes_dict['historical_bitcoin_prices'])

## Calculate Process

In [ ]:
def moving_average(df1):
    try:
        # Definir ventana para promedio móvil de 5 días
        window_spec = Window.orderBy('date').rowsBetween(-4, 0)

        mavg_df = df1.withColumn('5_day_moving_avg', avg(col('prices')).over(window_spec))

        return mavg_df

    except Py4JJavaError as err:
        if 'org.apache.hadoop.security.AccessControlException' in str(err.java_exception):
            print('Permission denied: User does not have access to the specified file or directory.')
        else:
            print('Unexpected Py4JJavaError:', str(err))

    except AnalysisException as err:
        if 'Permission denied' in str(err):
            print('Permission denied: User does not have access to the specified file or directory.')
        elif 'cannot resolve' in str(err):
            print('Column not found: The specified column does not exist in the table.')
        elif 'Table or view not found' in str(err):
            print('Table not found: The specified table or view does not exist.')
        else:
            print('An AnalysisException occurred:', str(err))

    except Exception as err:
        print('An unexpected error occurred:', str(err))

    return None

In [ ]:
df_moving_avg = moving_average(df1=df_transformed['historical_bitcoin_prices_transf'])

# Load Stage

In [ ]:
# Define path for write process
OUTPUT_PATH = 'data/output/'

# Define table_names
CRYPTOS_TABLE =  'cryptos'
MARKET_BITCOIN_TABLE = 'market_bitcoin'
HIST_BITCOIN_PRICES_TABLE = 'hist_bitcoin_prices'
MOVING_AVERAGE_BITCOIN_TABLE = 'moving_average_bitcoin'

In [ ]:
# Implement repartitio method
cryptos = dataframes_dict['cryptos'].repartition(10)
market_bitcoin = df_transformed['market_bitcoin_transf']
historical_bitcoin_prices = df_transformed['historical_bitcoin_prices_transf'].repartition(10)
moving_average_bitcoin = df_moving_avg.repartition(10)

In [ ]:
def write_parquet(df, path, partition_col):
    try:
        df.write.option('partitionOverwriteMode', 'dynamic')\
            .mode('overwrite')\
            .partitionBy(partition_col)\
            .parquet(path)

        print(f"Successfully written: {path}")

    except Py4JJavaError as err:
        if 'org.apache.hadoop.security.AccessControlException' in str(err.java_exception):
            print('Permission denied: User does not have access to the specified file or directory.')
        else:
            print('Unexpected Py4JJavaError:', str(err))

    except AnalysisException as err:
        if 'Permission denied' in str(err):
            print('Permission denied: User does not have access to the specified file or directory.')
        elif 'cannot resolve' in str(err):
            print('Column not found: The specified column does not exist in the table.')
        elif 'Table or view not found' in str(err):
            print('Table not found: The specified table or view does not exist.')
        else:
            print('An AnalysisException occurred:', str(err))

    except Exception as err:
        print('An unexpected error occurred:', str(err))

    return None

In [ ]:
# Write each DataFrame
write_parquet(cryptos, f'{OUTPUT_PATH}{CRYPTOS_TABLE}', 'id')
write_parquet(market_bitcoin, f'{OUTPUT_PATH}{MARKET_BITCOIN_TABLE}', 'crypto_id')
write_parquet(historical_bitcoin_prices,f'{OUTPUT_PATH}{HIST_BITCOIN_PRICES_TABLE}', 'date')
write_parquet(moving_average_bitcoin, f'{OUTPUT_PATH}{MOVING_AVERAGE_BITCOIN_TABLE}', 'date')